# Desarrollo de Transformadores y Pipelines de Procesamiento 

En esta sesión, exploraremos cómo estructurar y automatizar el preprocesamiento de datos mediante el uso de Transformadores y Pipelines. Continuaremos utilizando el conjunto de datos **NSL-KDD**, el cual es un estándar en la evaluación de Sistemas de Detección de Intrusiones (IDS).

El dataset NSL-KDD clasifica las conexiones de red en dos grandes grupos: **tráfico normal** y **ataques cibernéticos**. Esta base de datos superó las limitaciones de su predecesor (KDD'99) al eliminar registros redundantes, lo que evita el sesgo en los algoritmos de clasificación.

A pesar de su antigüedad, sigue siendo una herramienta pedagógica y de investigación invaluable debido a la limpieza de sus etiquetas y su volumen de información manejable.

### Acceso a los datos
El repositorio oficial utilizado se encuentra en: **https://www.kaggle.com/datasets/hassan06/nslkdd**.

## 0. Incorporación de Módulos y Librerías
Cargamos las herramientas necesarias para la manipulación de los datos, así como las clases específicas de Scikit-Learn para construir nuestros flujos de procesamiento.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

#### Herramienta Auxiliar de División
Definimos una función práctica para realizar la separación de los datos en tres bloques: entrenamiento, validación y evaluación, permitiendo la estratificación si es necesaria.

In [5]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    
    return train_set, val_set, test_set

## 1. Obtención de la Información
Leemos el archivo CSV previamente procesado.

In [6]:
df = pd.read_csv("KDDTrain+.csv")
df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


### Fragmentación del Dataset
Aplicamos la función auxiliar para generar nuestros tres conjuntos de datos, asegurando que la proporción de protocolos de red se mantenga balanceada mediante estratificación.

In [7]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

print(f"Dimensiones Entrenamiento: {train_set.shape}")
print(f"Dimensiones Validación: {val_set.shape}")
print(f"Dimensiones Prueba: {test_set.shape}")

Dimensiones Entrenamiento: (75583, 43)
Dimensiones Validación: (25195, 43)
Dimensiones Prueba: (25195, 43)


## 2. Preparación y Transformaciones de Limpieza
Antes de automatizar el proceso, extraemos nuestra variable dependiente y aplicamos un tratamiento inicial a ciertos valores anómalos.

In [8]:
# Aislamos las variables predictoras (X) de la etiqueta objetivo (y)
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [9]:
# Manejo manual de valores atípicos (outliers)
# Asignamos valores nulos a ciertos rangos de bytes para que luego sean procesados por un imputador
X_train.loc[(X_train["src_bytes"] > 400) & (X_train["src_bytes"] < 800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"] > 500) & (X_train["dst_bytes"] < 2000), "dst_bytes"] = np.nan

X_train.head(5)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.0,0.0,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.0,1.0,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.0,0.0,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.0,1.0,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.0,0.0,0.0,0.0,16


## 3. Ensamblaje de Tuberías de Procesamiento (Pipelines)
El uso de la clase `Pipeline` de Scikit-Learn permite concatenar secuencias de transformaciones. Esto garantiza que exactamente el mismo preprocesamiento que se aplicó al conjunto de entrenamiento se replique sin errores en los datos de prueba o en entornos de producción.

**Aspectos fundamentales:**
* Se estructuran mediante una lista de tuplas donde cada elemento es `('identificador', instancia_del_estimador)`.
* Todos los eslabones de la cadena deben contener los métodos `fit` y `transform` (excepto el último, que podría ser solo un modelo predictivo).
* Al invocar `.fit()`, los datos fluyen siendo transformados secuencialmente paso a paso.

In [10]:
# Diseñamos un flujo exclusivo para variables numéricas
# 1. Rellenamos nulos con la mediana
# 2. Escalamos los datos haciéndolos resistentes a valores extremos
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('rbst_scaler', RobustScaler())
])


Para aplicar diferentes procesos según el tipo de variable (numérica o texto), emplearemos un `ColumnTransformer`. Este unirá nuestro pipeline numérico y nuestro codificador categórico en una única operación central.

In [11]:
# Identificamos los nombres de las columnas según su tipo de dato
num_attribs = list(X_train.select_dtypes(exclude=['object']))
cat_attribs = list(X_train.select_dtypes(include=['object']))

# Consolidamos las transformaciones
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs)
])


/tmp/ipykernel_11904/3425913876.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_attribs = list(X_train.select_dtypes(include=['object']))


In [12]:
# Ejecutamos el flujo de preprocesamiento completo sobre nuestros datos de entrenamiento
X_train_prep = full_pipeline.fit_transform(X_train)

In [13]:
# Reconstruimos el DataFrame con las nuevas características generadas (incluyendo el One-Hot Encoding)
X_train_prep = pd.DataFrame(
    X_train_prep, 
    columns=list(pd.get_dummies(X_train)), 
    index=X_train.index
)

X_train_prep.head(10)

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
113467,0.0,0.000000,235.718062,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
31899,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
108116,0.0,1.056680,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
89913,0.0,-0.174089,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
106319,0.0,-0.141700,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
98007,0.0,0.012146,0.612335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
16447,0.0,7.072874,1.599119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
64957,1.0,0.000000,1.449339,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
100052,0.0,0.659919,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
28800,0.0,1.178138,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Llegados a este punto, la matriz `X_train_prep` se encuentra completamente saneada, escalada y codificada. Está en el formato matemático ideal para alimentar cualquier algoritmo de Machine Learning.